## Step 1: Imports and reproducibility


In [ ]:
import sys, time, random
from pathlib import Path

root = Path.cwd().parent
sys.path.append(str(root))
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import json

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import plot_model

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import src.training.pretraining as Pretrain
from src.models.cnn import build_model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

I0000 00:00:1791994390.518767    8418 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791994390.524562    8418 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791994390.977490    8418 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791994392.911119    8418 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

TensorFlow: 2.21.0
GPUs: []
TensorFlow: 2.21.0


E0000 00:00:1791994393.599823    8418 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## Step 2: Load the preprocessed data


In [2]:
X_train, X_test, y_train, y_test, y_train_cat, y_test_cat, encoder, class_names = Pretrain.load()

X_train: (4916, 224, 224, 3) | y_train_cat: (4916, 4)
X_test : (2107, 224, 224, 3) | y_test_cat : (2107, 4)
Classes: [np.str_('glioma'), np.str_('meningioma'), np.str_('notumor'), np.str_('pituitary')]
Pixel range: 0.0 1.0


## Sanity check

In [3]:
Pretrain.sanityCheck(X_train, y_train_cat, y_train, y_test)

Train class counts: [1135 1151 1400 1230]
Test class counts : [486 494 600 527]
All checks passed


## Create a validation set

In [ ]:
X_tr, X_val, y_tr, y_val, y_tr_int, y_val_int = train_test_split(
    X_train, y_train_cat, y_train,
    test_size=0.15,
    stratify=y_train,
    random_state=SEED,
)

print("Train     :", X_tr.shape, y_tr.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test      :", X_test.shape, y_test_cat.shape)

## 4. Hyperparameters

These defaults are used unless `tuning.ipynb` saved better ones.

In [ ]:
params = {"learning_rate": 1e-3, "dropout": 0.5, "dense_units": 256, "batch_size": 32}
EPOCHS = 30   # upper limit: early stopping usually stops sooner

hp_file = root / "models" / "best_hparams.json"
if hp_file.exists():
    params.update(json.loads(hp_file.read_text()))
    print("Using tuned hyperparameters")

print(params)

## 5. Build and compile the model

The architecture is in `src/models/cnn.py`: 4 blocks of Conv2D + MaxPooling, then a Dense classifier with Dropout.
- Hidden layers use **ReLU**.
- The output layer uses **softmax** (4 probabilities that sum to 1).
- Compiled with the **Adam** optimizer and **categorical_crossentropy** loss.

In [ ]:
model = build_model(params["learning_rate"], params["dropout"], params["dense_units"])
model.summary()

In [ ]:
# Needs pydot and Graphviz installed
try:
    plot_model(model, to_file=str(root / "model.png"), show_shapes=True)
except Exception:
    print("plot_model skipped (install pydot and Graphviz to use it)")

## 6. Callbacks

- **ModelCheckpoint** saves the best model (lowest validation loss).
- **EarlyStopping** stops training when validation loss stops improving, and goes back to the best weights.

In [ ]:
models_dir = root / "models"
models_dir.mkdir(exist_ok=True)

callbacks = [
    ModelCheckpoint(str(models_dir / "best_model.h5"), monitor="val_loss", save_best_only=True),
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
]

## 7. Train (with timing)

In [ ]:
start = time.time()

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=params["batch_size"],
    callbacks=callbacks,
)

print(f"Training time: {(time.time() - start) / 60:.1f} min")

## 8. Learning curves

If training gets better while validation gets worse, the model is overfitting.

In [ ]:
plt.figure(figsize=(12, 4))

for i, metric in enumerate(["accuracy", "loss"]):
    plt.subplot(1, 2, i + 1)
    plt.plot(history.history[metric], label="train")
    plt.plot(history.history["val_" + metric], label="validation")
    plt.title(metric)
    plt.xlabel("epoch")
    plt.legend()

plt.tight_layout()
plt.show()

## 9. Evaluate on the test set

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test_cat)
print(f"Test accuracy: {test_acc:.4f}")

## 10. Classification report and confusion matrix

In [ ]:
probs = model.predict(X_test)
y_pred = np.argmax(probs, axis=1)

print(classification_report(y_test, y_pred, target_names=class_names))

sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

## 11. Correct and incorrect predictions

In [ ]:
def show(indices, title):
    plt.figure(figsize=(12, 3.5))
    for i, idx in enumerate(indices[:4]):
        plt.subplot(1, 4, i + 1)
        plt.imshow(X_test[idx])
        plt.title(f"True: {class_names[y_test[idx]]}\nPred: {class_names[y_pred[idx]]}")
        plt.axis("off")
    plt.suptitle(title)
    plt.show()

show(np.where(y_pred == y_test)[0], "Correct predictions")
show(np.where(y_pred != y_test)[0], "Incorrect predictions")

## 12. Save the model

In [ ]:
model.save(str(models_dir / "brain_tumor_cnn.h5"))
joblib.dump(class_names, models_dir / "class_names.pkl")   # needed by the API

print("Saved to", models_dir)